# 2.6 — Exploratory Data Analysis (EDA)

EDA is what you do before any modelling. You look at the data from every angle to understand its shape, spot problems, find patterns, and decide what cleaning and engineering it needs.

> **Why it matters:** Without EDA you throw raw data at a model, get poor results, and have no idea why. With EDA you know exactly what the data needs before writing a single model line.

### EDA connects everything from 2.1–2.5:
- Missing values (2.1) — found during EDA
- Outliers (2.2) — spotted in boxplots and describe()
- Encoding needs (2.3) — identified by checking dtypes
- Scaling needs (2.4) — identified by checking ranges
- Feature ideas (2.5) — inspired by groupby and correlation findings

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Load Titanic — no download needed, seaborn has it built in
df = sns.load_dataset('titanic')
print("Dataset loaded:", df.shape)
df.head()

## Step 1 — Shape & Structure

In [ ]:
print("Shape:", df.shape)
print("\nColumns:", df.columns.tolist())

In [ ]:
# Data types and null counts — most important first command
df.info()

In [ ]:
# Missing values
missing = df.isnull().sum().sort_values(ascending=False)
missing_pct = (df.isnull().sum() / len(df) * 100).sort_values(ascending=False)
pd.DataFrame({'count': missing, 'percent': missing_pct.round(1)}).head(10)

In [ ]:
# Duplicates
print("Duplicate rows:", df.duplicated().sum())

# Unique values per column
df.nunique().sort_values()

## Step 2 — Statistics & Distributions

In [ ]:
# Summary stats for numeric columns
df.describe().round(2)

In [ ]:
# Skewness — high value means skewed
df.skew(numeric_only=True).sort_values(ascending=False).round(2)

In [ ]:
# Class imbalance — how balanced is the target?
print("Survival counts:")
print(df['survived'].value_counts())
print("\nSurvival rate:")
print((df['survived'].value_counts(normalize=True) * 100).round(1))

## Step 3 — Visualisations

In [ ]:
# Histograms for numeric columns
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

sns.histplot(df['age'].dropna(), kde=True, ax=axes[0])
axes[0].set_title('Age distribution')

sns.histplot(df['fare'], kde=True, ax=axes[1])
axes[1].set_title('Fare distribution')

sns.histplot(df['sibsp'], kde=False, ax=axes[2])
axes[2].set_title('Siblings/Spouses')

plt.tight_layout()
plt.show()

In [ ]:
# Boxplots — spot outliers
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

sns.boxplot(x=df['fare'], ax=axes[0])
axes[0].set_title('Fare — outliers?')

sns.boxplot(x='survived', y='age', data=df, ax=axes[1])
axes[1].set_title('Age by survival')

plt.tight_layout()
plt.show()

In [ ]:
# Countplots for categorical columns
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

sns.countplot(x='sex', hue='survived', data=df, ax=axes[0])
axes[0].set_title('Sex vs Survived')

sns.countplot(x='pclass', hue='survived', data=df, ax=axes[1])
axes[1].set_title('Pclass vs Survived')

sns.countplot(x='embarked', hue='survived', data=df, ax=axes[2])
axes[2].set_title('Embarked vs Survived')

plt.tight_layout()
plt.show()

## Step 4 — Correlations & Relationships

In [ ]:
# Correlation with target
df.corr(numeric_only=True)['survived'].sort_values().round(3)

In [ ]:
# Correlation heatmap
plt.figure(figsize=(8, 6))
sns.heatmap(
    df.corr(numeric_only=True),
    annot=True, fmt='.2f', cmap='coolwarm', center=0
)
plt.title('Correlation matrix')
plt.show()

In [ ]:
# Groupby — survival rate by categorical features
print("Survival rate by sex:")
print(df.groupby('sex')['survived'].mean().round(2))

print("\nSurvival rate by pclass:")
print(df.groupby('pclass')['survived'].mean().round(2))

print("\nSurvival rate by embarked:")
print(df.groupby('embarked')['survived'].mean().round(2))

## Step 5 — EDA Findings & Conclusions

Write your observations here after running the cells above.

**Missing values:**
- Age: ~20% missing → fill with median
- Deck/Cabin: ~77% missing → drop column
- Embarked: 2 missing → fill with mode

**Outliers:**
- Fare has extreme outliers (max 512 vs 75th percentile 31) → cap or log transform

**Class imbalance:**
- 62% did not survive, 38% survived → accuracy alone is misleading

**Key patterns found:**
- Women survived 74% vs men 19% → sex is a strong feature
- 1st class survived 63% vs 3rd class 24% → pclass is important
- Higher fare = higher survival → fare correlates with pclass

**Encoding needed:**
- sex → label encode (binary)
- embarked → one-hot encode
- pclass → already numeric, keep as is

**Feature engineering ideas:**
- family_size = sibsp + parch + 1
- is_alone = 1 if family_size == 1
- title extracted from name (Mr, Mrs, Miss, Master)

## The 14-Step EDA Checklist

Run this on every new dataset:

| # | Command | What it tells you |
|---|---------|------------------|
| 1 | `df.shape` | Rows and columns |
| 2 | `df.info()` | Types and nulls |
| 3 | `df.isnull().sum()` | Missing value counts |
| 4 | `df.duplicated().sum()` | Duplicate rows |
| 5 | `df.describe()` | Stats — check min/max for outliers |
| 6 | `df.skew()` | Which columns are skewed |
| 7 | `df['target'].value_counts(normalize=True)` | Class imbalance |
| 8 | `df.nunique()` | Which columns are categorical |
| 9 | Histograms | Shape of distributions |
| 10 | Boxplots | Outliers visually |
| 11 | Countplots | Frequency of categories |
| 12 | `df.corr()['target']` | Features related to target |
| 13 | `groupby('col')['target'].mean()` | Category vs target |
| 14 | Correlation heatmap | Multicollinearity |

## Practice Task

Run the full 14-step checklist on the Titanic dataset and write your own observations in a markdown cell.

In [ ]:
# YOUR EDA HERE — work through all 14 steps
# Write observations as markdown cells below each finding